In [ ]:
#TMDB Extract

In [ ]:
from src.etl.extract import Extract

extractor = Extract()
data = extractor.fetch_movies(50)

len(data)
data[0]

Basic Transform

In [ ]:
from src.etl.transform import Transform

df = Transform.to_dataframe(data)

print(df.shape)
df.head()
df.describe()

In [ ]:
df.isnull().sum()

In [ ]:
df[df["release_date"].isnull()]

In [ ]:
df.groupby("release_year").size().plot(title="Movies per Year")

In [ ]:
df.sort_values(by="rating", ascending=False).head(10)[["title", "rating"]]

In [ ]:
import os
from dotenv import load_dotenv
import duckdb
from src.utils.paths import project_path

load_dotenv()
DUCKDB_PATH_ENV = os.getenv("DUCKDB_PATH", "data/db/movies.duckdb")
DB_PATH = project_path(DUCKDB_PATH_ENV)
print("Resolved DuckDB path:", DB_PATH)

# Connect and query
with duckdb.connect(DB_PATH) as con:
    df_db = con.execute("SELECT * FROM movies").fetchdf()
    df_db.head()

##Upcoming movies

In [ ]:
from datetime import datetime

today = datetime.today().date()

unreleased = df[df["release_date"] > today]
unreleased.head()

Test case för top5 of movies

In [ ]:
from src.etl.extract import Extract
from src.services.tmdb_service import TMDBService

service = TMDBService()
movies = extractor.fetch_movies(pages=25)

# Only enrich top X
for movie in movies[:10]:
    details, status = service.get_movie_details(movie["id"])
    print(details)

In [ ]:
from src.repository.movie_repository import MovieRepository

with MovieRepository(DB_PATH) as repo:
    print(repo.conn.execute("SHOW TABLES").fetchall())


In [ ]:
for movie in movies[:1]:
    details, status = service.get_movie_details(movie["id"])

    print(
        details.movie_id,
        details.runtime,
        details.director
    )

Grab from the database so we can include cast and crew.

In [ ]:
# top 5 from the cached database including full cast and crew
for movie in movies[:5]:
    details = service.repo.get_full_movie_details(movie["id"])
    print(details)

In [ ]:
repo.close()
con.close()